# Introduction

The notebook for calculating intrument functions.
For more details see signal_processing_pipeline.ipynb.

# Libraries


In [ ]:
from datetime import datetime
import numpy as np
from scipy.signal import find_peaks, peak_widths
from scipy.stats import linregress
from scipy.optimize import curve_fit
from lmfit.models import SplineModel, SkewedGaussianModel
from lib.file_func import get_instrument_type, get_sum_signal, load_coord
from lib.peak import fwhm_to_sigma, detect_peaks, gen_peak
from backend.api_wrapper import mascope_api
from matplotlib import pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

# Functions


In [ ]:
def fit_gaussian(x: np.array, y: np.array):
    """Fit the spectrum range with a skewed Gaussian peak-shape using lmfit

    :param x: mz scale
    :type x: array-like
    :param y: counts
    :type y: array-like
    :return: ModelResult (see lmfit doc)
    :rtype: ModelResult
    """
    # Initialize fitting parameters for the main peak
    model = SkewedGaussianModel(prefix="p_")
    params = model.make_params()
    if instrument_type == "tof":
        # Fitting parameters for the background
        knot_xvals = np.array([-dmz, -dmz / 2, -dmz / 3, dmz / 3, dmz / 2, dmz])
        bkg = SplineModel(prefix="bkg_", xknots=knot_xvals)
        params.update(bkg.guess(y, x))
        # Total model
        model = model + bkg

    params.add("p_amplitude", value=1, max=1.1)
    params.add("p_center", value=0)
    params.add("p_sigma", value=0.01)
    params.add("p_gamma", value=0.1)

    # Perform fitting
    fit = model.fit(y, params, x=x)
    return fit


def calculate_fwhm(x, y):
    # Find the maximum count and its index
    max_index = np.argmax(y)
    max_count = y[max_index]

    # Find the half maximum count
    half_max_count = max_count / 2

    # Find the indices where the counts are closest to the half maximum
    left_index = np.argmin(np.abs(y[:max_index] - half_max_count))
    right_index = np.argmin(np.abs(y[max_index:] - half_max_count)) + max_index

    # Calculate the FWHM
    fwhm = x[right_index] - x[left_index]

    return fwhm


def calculate_center_of_mass(x, y):
    # Calculate the center of mass
    center_of_mass = np.sum(x * y) / np.sum(y)
    return center_of_mass

# Load file and find peaks

If too many peaks are detected, increase `q`  parameter in `np.percentile` (or decrease otherwise). 

In [ ]:
# Load data file
filename = "TestKLTOF1_2023.11.09_12h47m34s_-"
# Define the MS
instrument_type = get_instrument_type(filename)
# Extract averaged spectrum and mz values
spec = get_sum_signal(filename).values
mz = load_coord(filename, "sum_signal", "mz")
# Find peaks. Tune the threshold if there are no peaks/too many peaks detected.
peak, _ = find_peaks(spec, height=np.percentile(spec, q=99.0))
fwhm = peak_widths(spec, peak, rel_height=0.5)[0]
sig = np.array([fwhm_to_sigma(fw) for fw in fwhm])
# Plot result
fig = go.Figure(
    [
        go.Scatter(x=mz, y=spec, name="Raw counts"),
        go.Scatter(x=mz[peak], y=spec[peak], mode="markers", name="Detected peaks"),
    ]
)
fig.update_layout(
    height=300, width=800, margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4)
)
print(f"{len(peak)} peaks were detected.")
fig.show()

# Calculate peak shape

Expected half-width band presumably containing one peak. 
Decrease `dmz` if you get too many peaks in one band.
Increase `dmz` if the peak shape is distorted/cut.

In [ ]:
# TODO find way to automatically estimate optimal dmz value
dmz = 0.1
r_squared_threshold = 0.96

peak_traces = []
peak_norm_traces = []

p_x = np.linspace(-10, 10, 101)
p_ys = []
p_mzs = []
p_fwhms = []
p_centers = []

for i, p in enumerate(tqdm(peak)):
    p_height = spec[p]

    # Select a narrow region (peak center +/- dmz) of the spectrum around the peak
    p_mz_center = mz[p]
    p_mz0 = p_mz_center - dmz
    p_mz1 = p_mz_center + dmz
    # Find indices
    p_i0 = np.argmin(np.abs(mz - p_mz0))  # left border
    p_i1 = np.argmin(np.abs(mz - p_mz1))  # right border
    p_i = p - p_i0  # center
    # Peak region
    p_mz = mz[p_i0:p_i1]
    p_spec = spec[p_i0:p_i1]

    if np.max(p_spec) > p_height:
        # 'p' is not the biggest peak in range, dismiss
        continue

    # Normalize peak region: mz around 0 and spec to range [0, 1]
    p_mz_norm = p_mz - p_mz_center
    p_spec_norm = p_spec / p_height
    p_spec_norm -= p_spec_norm.min()
    peak_traces.append(go.Scatter(x=p_mz_norm, y=p_spec_norm))

    # Fit peak in the region
    fit = fit_gaussian(p_mz_norm, p_spec_norm)
    p_spec_norm_fit = fit.eval_components()["p_"]

    if fit.rsquared < r_squared_threshold:
        continue  # fitting error to large, dismiss

    # Remove junk peaks arond main one
    mask = np.where(fit.best_fit < 1e-9)
    p_spec_norm[mask] = 0

    # Get peak top location
    if instrument_type == "tof":
        # Remove baseline if TOF
        p_spec_norm -= fit.eval_components()["bkg_"]
        p_spec_norm[np.where(p_spec_norm < 0)] = 0

    top_y = np.max(p_spec_norm_fit)
    top_x = calculate_center_of_mass(p_mz_norm, p_spec_norm_fit)

    # Refine peak position and height
    p_mz_norm -= top_x
    p_spec_norm /= top_y

    # Get and store Gaussian peak sigma and width
    try:
        p_fwhm = calculate_fwhm(p_mz_norm, p_spec_norm_fit)
    except Exception:
        continue

    p_sigma = p_fwhm / (2 * np.sqrt(2 * np.log(2)))
    # Scale peak to width sigma=1
    p_mz_norm /= p_sigma
    # Interpolate the normalized (both width and height) peak into predefined domain "p_x"
    p_y = np.interp(p_x, p_mz_norm, p_spec_norm, left=0, right=0)

    if np.all(np.isnan(p_y)):
        continue

    # Store peak positions, widths, and refined peak shape
    p_mzs.append(p_mz_center)
    p_centers.append(top_x)
    p_fwhms.append(p_fwhm)
    p_ys.append(p_y)
    peak_norm_traces.append(go.Scatter(x=p_x, y=p_y, name=f"peak {i}"))


# Clean shifted outliers
p_centers = np.array(p_centers)
non_outlier_indx = np.where(p_centers - np.median(p_centers) < 1e-3)[0]
peak_norm_traces = [peak_norm_traces[i] for i in non_outlier_indx]
p_fwhms = [p_fwhms[i] for i in non_outlier_indx]
p_ys = [p_ys[i] for i in non_outlier_indx]
p_mzs = [p_mzs[i] for i in non_outlier_indx]

# Calculate median peak shape
p_median = np.median(np.array([p_y for p_y in p_ys]), axis=0)
# Check if p_median is empty

if p_median.all() == np.nan:
    raise Exception(
        """p_median is empty
        Probably fitting error threshold is too strict"""
    )

# Add to the figure
peak_norm_traces.append(
    go.Scatter(x=p_x, y=p_median, line={"color": "red", "width": 3}, name="median")
)
# Plot the output
fig = go.FigureWidget(
    peak_norm_traces,
    {"title": f"Median peak shape of {len(peak_norm_traces)-1} peaks"},
)
fig.update_layout(
    height=300, width=800, margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4)
)


def toggle_trace_visibility(trace, points, selector):
    """Updates the median peak shape based on shapes present in the figure"""
    global p_median
    if not points.point_inds:
        return

    trace.visible = "legendonly"
    selected_traces = [
        trace.y
        for trace in fig.data
        if ("peak" in trace.name and trace.visible != "legendonly")
    ]
    p_median = np.median(np.array(selected_traces), axis=0)

    fig.update_layout(
        title=f"Median peak shape of {len(selected_traces)} peaks",
        height=300,
        width=800,
        margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4),
    )
    fig.update_traces(
        {"y": p_median},
        selector=lambda trace: trace.name == "median",
    )


print("Click on distorted peaks to remove them from peak shape estimation.")

[trace.on_click(toggle_trace_visibility) for trace in fig.data]
fig

Save median peak shape


In [ ]:
# Save peak
peak_shape = {"x": np.array(p_x), "y": np.array(p_median)}
p_ys = np.array(p_ys)
# Calculate residuals for normalized peaks vs median peak shape
norm_residuals = p_ys - peak_shape["y"]
## calculate R-squared vs mz
r_sq_norm = 1 - (norm_residuals**2).sum(axis=1) / ((p_ys - p_ys.mean()) ** 2).sum(
    axis=1
)
# Plot error vs mz
err_fig = plt.figure(figsize=(7, 1.5))
plt.scatter(p_mzs, r_sq_norm)
plt.xlabel("mz")
plt.ylabel(r"R$^2$")

# Resolution function

## Support functions

In [ ]:
# Resolution functions
R_tof = lambda mz: mz / (tof_a * mz + tof_b)
R_orb = lambda mz: orb_a / np.sqrt(mz)
# Fittin support functions
line = lambda x, a, b: a * x + b
polynome = lambda x, a, b, c: a * x**2 + b * x + c
rational_polynome = lambda x, a, b: x / (a * x + b)
inverse_sqrt = lambda x, a: a / np.sqrt(x)


def update_chosen_peak(trace, points, selector):
    """Update chosen peak in resolution function visualization"""
    if points.xs:
        fig_widget.layout.shapes = []
        chosen_peak_val = points.xs[0]
        # Mask area around chosen_peak_val
        chosen_peak_mask = (chosen_peak_val - dmz < mz) & (mz < chosen_peak_val + dmz)
        # Filter spectra window to plot
        mz_window_x = mz[chosen_peak_mask]
        mz_window_y = spec[chosen_peak_mask]
        # Init fitted signal
        fit_y = np.zeros_like(mz_window_x)
        # Fitted peak mask
        fit_peak_mask = (chosen_peak_val - dmz < fit_poss) & (
            fit_poss < chosen_peak_val + dmz
        )
        fit_poss_filt = fit_poss[fit_peak_mask]
        fit_heis_filt = fit_heis[fit_peak_mask]
        # Add fitted peaks to fit_y
        for i, mz_val in enumerate(fit_poss_filt):
            fit_y += gen_peak(
                mz_window_x,
                mz_val,
                fit_heis_filt[i],
                pres=resolution_function(mz_val),
                ps=peak_shape,
            )
            fig_widget.add_shape(
                type="line",
                x0=mz_val,
                x1=mz_val,
                y0=0,
                y1=fit_heis_filt[i],
                line=dict(width=2),
                xref="x2",
                yref="y2",
            )

            # fig_widget.add_vline(
            #     mz_val, y0=0, y1=fit_heis_filt[i], row=1, col=2, yref="y1"
            # )
        # Residuals
        residuals = mz_window_y - fit_y

        fig_widget.update_traces(
            {"x": mz_window_x, "y": mz_window_y},
            selector=lambda trace: trace.name == "Chosen peak",
        )
        fig_widget.update_traces(
            {"x": mz_window_x, "y": fit_y},
            selector=lambda trace: trace.name == "Fitted signal",
        )
        fig_widget.update_traces(
            {"x": mz_window_x, "y": residuals},
            selector=lambda trace: trace.name == "Residuals",
        )

## Evaluate coefficients for the resolution function

If calculation is failing, the initial guesses are returned in case of TOF and 1.725e6 for Orbitrap. 

In [ ]:
p_mzs = np.array(p_mzs)
# Get the fitted line depending on the instrument
if instrument_type == "tof":
    regres = linregress(p_mzs, p_fwhms)
    p_fwhms_line = line(p_mzs, regres.slope, regres.intercept)
if instrument_type == "orbi":
    coefs = np.polyfit(p_mzs, p_fwhms, 2)
    p_fwhms_line = polynome(p_mzs, *coefs)

# Points higher than ndev * std_dev will be later filtered out
ndev = 1
# Get residuals and standard deviation
residuals = p_fwhms - p_fwhms_line
std_dev = np.std(residuals)
is_outlier = (residuals > ndev * std_dev) | (residuals < -ndev * std_dev)
# Remove outliers
p_fwhms_filt = np.array(p_fwhms, dtype=np.double)[~is_outlier]
mass = np.array(p_mzs, dtype=np.double)[~is_outlier]
resolution = mass / p_fwhms_filt

if instrument_type == "tof":
    # Fit resolution with the inverse square root
    try:
        # Initial guesses
        a_init = 1 / np.mean(resolution)
        b_init = np.mean(resolution / mass)
        popt, pcov = curve_fit(rational_polynome, mass, resolution, p0=(a_init, b_init))
        tof_a, tof_b = popt
    except ValueError:
        print("No points available to fit resolution function")
        tof_a, tof_b = a_init, b_init
    resolution_function = R_tof

    print(f"TOF, a={tof_a:.2e}, b={tof_b:.2e}")

if instrument_type == "orbi":
    # Fit resolution with the inverse square root
    try:
        popt, pcov = curve_fit(inverse_sqrt, mass, resolution)
        orb_a = popt[0]
    except ValueError:
        print("No points available to fit resolution function")
        orb_a = 1.725e6
    resolution_function = R_orb

    print(f"Orbi, a={orb_a:.2e}")

## Fit peaks

In [ ]:
# Fit the peaks for the current resolution function and peak shape
sample_file_data = await detect_peaks(
    filename,
    (peak_shape, resolution_function),
    0.9,
    u_list=p_mzs,  # for testing
    max_n_peaks=5,
    if_exists="replace",
    dmz=0.5,
    instrument_type=instrument_type,
)

## Resolution function visualization

In [ ]:
# Set plotting width at chosen peak window
dmz = 0.1

# Get fitted peaks
fit_heis = sample_file_data.peak_heights.dropna(dim="mz").sum(dim="time").values
fit_poss = load_coord(filename, "peak_heights", "mz")

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[[{}, {}], [{"colspan": 2}, None]],
    subplot_titles=("FWHM", "Chosen peak", "Resolution function"),
)

# FWHM
fig.add_traces(
    [
        go.Scatter(x=p_mzs, y=p_fwhms, mode="lines", name="True FWHM"),
        go.Scatter(
            x=p_mzs,
            y=p_fwhms_line,
            mode="lines",
            line=dict(dash="dash"),
            name="Approximation",
        ),
        go.Scatter(
            x=np.concatenate([p_mzs, p_mzs[::-1]]),
            y=np.concatenate(
                [p_fwhms_line + ndev * std_dev, (p_fwhms_line - ndev * std_dev)[::-1]]
            ),
            fill="toself",
            fillcolor="rgba(120, 120, 120, 0.2)",
            line=dict(color="rgba(0, 0, 0, 0)"),
            showlegend=False,
        ),
    ],
    rows=1,
    cols=1,
)
# FITTED RESOLUTION FUNCTION
mass_range = np.linspace(min(p_mzs), max(p_mzs), 100)
fig.add_traces(
    [
        go.Scatter(
            x=mass_range,
            y=resolution_function(mass_range),
            mode="lines",
            line=dict(color="red"),
            name="Fitted resolution function",
        ),
        go.Scatter(
            x=p_mzs,
            y=p_mzs / p_fwhms,
            mode="markers",
            marker=dict(color="grey"),
            name="Omitted pairs",
        ),
        go.Scatter(
            x=mass,
            y=resolution,
            mode="markers",
            marker=dict(color="black"),
            name="Used mass/resolution pairs",
        ),
    ],
    rows=2,
    cols=1,
)
# CHOSEN PEAK
chosen_peak_trace = go.Scatter(
    x=[0], y=[0], name="Chosen peak", line=dict(color="coral")
)
fit_signal_trace = go.Scatter(
    x=[0],
    y=[0],
    name="Fitted signal",
    line=dict(color="steelblue", width=4, dash="dash"),
)
residuals_trace = go.Scatter(
    x=[0],
    y=[0],
    name="Residuals",
    fill="tozeroy",
    fillcolor="rgba(70, 130, 180, 0.5)",
    line=dict(color="rgba(0, 0, 0, 0)"),
)
fig.add_traces([chosen_peak_trace, fit_signal_trace, residuals_trace], rows=1, cols=2)
vlines = []
# Update layout
fig.update_xaxes(title_text="mz", row=1, col=1)
fig.update_yaxes(title_text="FWHM", row=1, col=1)

fig.update_xaxes(title_text="mz", row=1, col=2)
fig.update_yaxes(title_text="Counts", row=1, col=2)

fig.update_xaxes(title_text="mz", row=2, col=1)
fig.update_yaxes(title_text="Resolution", row=2, col=1)

fig.update_layout(
    height=450, width=1000, margin=go.layout.Margin(l=30, r=30, b=30, t=30)
)

fig_widget = go.FigureWidget(fig)

fig_widget.data[4].on_click(update_chosen_peak)
fig_widget.data[5].on_click(update_chosen_peak)

fig_widget

# Save instrument functions


In [ ]:
# Get the current UTC date and time
datetime_utc = datetime.now()
# Convert the datetime object to a string in UTC format
datetime_utc_str = datetime_utc.isoformat()

mascope_api.create_instrument_function(
    mascope_url="http://0.0.0.0:8090",
    instrument="KORBI2",
    datetime_utc=datetime_utc_str,
    peakshape=peak_shape,
    resolution_function=resolution_function,
)